In [16]:
import os
import time
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

groq_key = os.getenv("GROQ_API_KEY")
assert groq_key, "GROQ_API_KEY not found in .env"

groq_model = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
cloud_llm = ChatOpenAI(
    model=groq_model,
    api_key=groq_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)
print(f"Using Cloud Model via Groq: {groq_model}")

FINDINGS = [
    "Port 2121/tcp open with banner: 220 (vsFTPd 2.3.4)",
    "FTP service on port 2121 allows anonymous login: 230 Login successful.",
    "Port 8080/tcp HTTP response header: Server: Apache/2.2.8 (Ubuntu) DAV/2",
    "Port 8080/tcp HTTP response header: X-Powered-By: PHP/5.2.4-2ubuntu5.10",
]

kb_path = Path("cve_knowledge_base.md")
if not kb_path.exists():
    kb_path.write_text(
        """# CVE & CWE Knowledge Base

## vsFTPd 2.3.4 Backdoor (CVE-2011-2523 / CWE-78)
- Severity: Critical
- Reference: CVE-2011-2523, CWE-78 (Improper Neutralization of Special Elements used in an OS Command)
- Remediation: Replace the compromised vsftpd 2.3.4 binary with a verified package from a trusted repository, audit the host for unauthorized processes/listeners, and rotate credentials.

## Anonymous FTP Access (CWE-220, CWE-287)
- Severity: High
- Reference: CWE-220 (Storage of File With Sensitive Data Under FTP Root), CWE-287 (Improper Authentication)
- Remediation: Disable anonymous FTP authentication, enforce authenticated access or SFTP, and audit directories for exposed files.

## Apache HTTP Server 2.2.8 EOL & DAV (CVE-2017-9798, CVE-2014-0118, CVE-2013-6438)
- Severity: High
- Reference: CVE-2017-9798, CVE-2014-0118, CVE-2013-6438
- Remediation: Upgrade from end-of-life Apache 2.2.8 to a supported release, disable unused modules such as mod_dav, and set ServerTokens Prod.

## Legacy PHP 5.2.4 Exposure (CVE-2010-2225, CVE-2011-1398)
- Severity: High
- Reference: CVE-2010-2225, CVE-2011-1398
- Remediation: Upgrade from legacy PHP 5.2.4 to a supported PHP version, set expose_php = Off to suppress X-Powered-By headers, and audit input/deserialization handling.
""",
        encoding="utf-8",
    )

kb_text = kb_path.read_text(encoding="utf-8")
parts = kb_text.split("\n## ")
retrieved_blocks = [b.strip() for b in parts[1:] if b.strip()] if len(parts) > 1 else [kb_text.strip()]
grounded_context = "\n\n---\n\n".join(retrieved_blocks)
print(f"Loaded {len(FINDINGS)} findings and {len(retrieved_blocks)} KB sections from {kb_path}.")

Using Cloud Model via Groq: openai/gpt-oss-20b
Loaded 4 findings and 4 KB sections from cve_knowledge_base.md.


In [17]:
# --- STAGE 1: Ungrounded Chain (No KB Retrieval) ---
stage1_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a security analyst. Assess these findings from memory only."),
    ("user", "Findings:\n{findings}"),
])
stage1_chain = stage1_prompt | cloud_llm | StrOutputParser()

t0 = time.perf_counter()
stage1_out = stage1_chain.invoke({"findings": "\n".join(f"- {f}" for f in FINDINGS)})
t1_sec = round(time.perf_counter() - t0, 2)
print(f"\n=== STAGE 1 (UNGROUNDED — NO KB) [{t1_sec}s] ===\n{stage1_out}")

# --- STAGE 2: Grounded RAG Chain (With cve_knowledge_base.md Context) ---
stage2_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a defensive security analyst. Summarize each finding, severity, "
        "exact CVE/CWE reference, and fix using ONLY the provided KB context. "
        "Do not invent facts outside the context.",
    ),
    ("user", "Findings:\n{findings}\n\nKB Context:\n{context}"),
])
stage2_chain = stage2_prompt | cloud_llm | StrOutputParser()

t0 = time.perf_counter()
stage2_out = stage2_chain.invoke({
    "findings": "\n".join(f"- {f}" for f in FINDINGS),
    "context": grounded_context,
})
t2_sec = round(time.perf_counter() - t0, 2)
print(f"\n=== STAGE 2 (GROUNDED — RAG KB) [{t2_sec}s] ===\n{stage2_out}")


=== STAGE 1 (UNGROUNDED — NO KB) [2.66s] ===
**Security Assessment – Findings Summary**

| Service | Port | Version / Banner | Known Vulnerabilities (as of 2024) | Risk Rating | Immediate Impact |
|---------|------|------------------|------------------------------------|-------------|------------------|
| vsFTPd | 2121/tcp | 220 vsFTPd 2.3.4 | • CVE‑2011‑2523 – “Backdoor” (remote code execution via crafted packet).<br>• CVE‑2011‑2524 – “Anonymous login bypass” (allows privileged access).<br>• General “vsFTPd 2.x” is no longer maintained; many CVEs exist. | **High** | Remote code execution, privilege escalation, data exfiltration. |
| FTP (anonymous) | 2121/tcp | 230 Login successful | • Anonymous access exposes entire FTP root to unauthenticated users.<br>• Potential for data theft, upload of malicious files, or use as a staging point for further attacks. | **High** | Data leakage, malicious uploads, pivoting. |
| Apache HTTP Server | 8080/tcp | Apache/2.2.8 (Ubuntu) | • CVE‑2010‑2861

In [18]:
import pandas as pd
from langchain_ollama import ChatOllama


def benchmark_grounded(label: str, llm, cost_label: str, exposure_label: str):
    chain = stage2_prompt | llm | StrOutputParser()
    t0 = time.perf_counter()
    text = chain.invoke({
        "findings": "\n".join(f"- {f}" for f in FINDINGS),
        "context": grounded_context,
    })
    elapsed = round(time.perf_counter() - t0, 2)
    expected_refs = [
        "CVE-2011-2523", "CWE-220", "CWE-287",
        "CVE-2017-9798", "CVE-2014-0118", "CVE-2013-6438",
        "CVE-2010-2225", "CVE-2011-1398",
    ]
    matched = sum(1 for r in expected_refs if r in text)
    return {
        "Model": label,
        "KB Refs Matched": f"{matched}/{len(expected_refs)}",
        "Latency (s)": elapsed,
        "Cost": cost_label,
        "Data Exposure": exposure_label,
        "Response": text,
    }


comparison_rows = []

# 1) Local Hosted Model (Ollama on 127.0.0.1:11434)
local_model = os.getenv("OLLAMA_MODEL", "qwen2.5:3b")
try:
    local_llm = ChatOllama(
        model=local_model,
        base_url="http://127.0.0.1:11434",
        temperature=0,
    )
    comparison_rows.append(
        benchmark_grounded(
            f"Local Ollama ({local_model})",
            local_llm,
            "$0 (local hardware)",
            "Local only (no external API)",
        )
    )
except Exception as exc:
    print(f"Local Ollama skipped ({exc})")

# 2) Cloud API Model (Groq)
comparison_rows.append(
    benchmark_grounded(
        f"Groq API ({groq_model})",
        cloud_llm,
        "$0 Free Tier (no CC)",
        "External (Groq API endpoint)",
    )
)

df_bench = pd.DataFrame(comparison_rows)
display(df_bench[["Model", "KB Refs Matched", "Latency (s)", "Cost", "Data Exposure"]])

,Model,KB Refs Matched,Latency (s),Cost,Data Exposure
0,Local Ollama (qwen2.5:3b),8/8,6.23,$0 (local hardware),Local only (no external API)
1,Groq API (openai/gpt-oss-20b),0/8,2.56,$0 Free Tier (no CC),External (Groq API endpoint)


In [19]:
import asyncio
import subprocess
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = Path("recon_server.py").resolve()

# --- Before vs. After KB Coverage (Service Side) ---
# Update before_grounded_count to match how many of the 4 finding types your initial KB covered
# before you expanded cve_knowledge_base.md (e.g., 1/4 = 25% -> 4/4 = 100%).
total_service_findings = len(FINDINGS) if "FINDINGS" in globals() else 4
before_grounded_count = 1
after_grounded_count = 4

print("=== SERVICE KB COVERAGE (BEFORE vs. AFTER) ===")
print(
    f"Before expansion: {before_grounded_count}/{total_service_findings} "
    f"({round(100 * before_grounded_count / total_service_findings)}%) grounded"
)
print(
    f"After expansion:  {after_grounded_count}/{total_service_findings} "
    f"({round(100 * after_grounded_count / total_service_findings)}%) grounded "
    "(vsFTPd 2.3.4, anonymous FTP, Apache 2.2.8/DAV/2, PHP 5.2.4)"
)
print(
    "Remaining service gaps: Deeper exploitability confirmation (e.g., triggering vsFTPd backdoor) "
    "and host/config-level auditing beyond read-only recon/banner/service probes."
)


async def run_initial_recon(target: str = "127.0.0.1"):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER_PATH)],
    )
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            ftp = await session.call_tool("banner_grab", {"target": target, "port": 2121})
            anon_ftp = await session.call_tool("anonymous_ftp_check", {"target": target, "port": 2121})
            http = await session.call_tool("http_probe", {"target": target, "port": 8080})
            blocked = await session.call_tool("banner_grab", {"target": "8.8.8.8", "port": 80})
            return (
                ftp.content[0].text.strip(),
                anon_ftp.content[0].text.strip(),
                http.content[0].text.strip(),
                blocked.content[0].text.strip(),
            )


async def run_approved_nmap(target: str = "127.0.0.1", ports: str = "2121,8080"):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER_PATH)],
    )
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            res = await session.call_tool("nmap_service_scan", {"target": target, "ports": ports})
            return res.content[0].text


ftp_out, anon_out, http_out, guard_out = await run_initial_recon("127.0.0.1")
print("\n[Initial Probe - FTP Banner]:", ftp_out)
print("[Initial Probe - Anon FTP]:", anon_out)
print("[Initial Probe - HTTP]:", http_out.splitlines()[0])
print("[Scope Guard]:", guard_out)

proposed = "nmap_service_scan(target='127.0.0.1', ports='2121,8080')"
print(f"\n[APPROVAL GATE] Proposed action: {proposed}")
decision = input("Type 'APPROVE' to execute nmap_service_scan: ").strip()

if decision == "APPROVE":
    nmap_out = await run_approved_nmap("127.0.0.1", "2121,8080")
    print("\n[APPROVED — Nmap Output]:\n", nmap_out)
else:
    print("\n[CANCELLED — Scan not approved.]")

=== SERVICE KB COVERAGE (BEFORE vs. AFTER) ===
Before expansion: 1/4 (25%) grounded
After expansion:  4/4 (100%) grounded (vsFTPd 2.3.4, anonymous FTP, Apache 2.2.8/DAV/2, PHP 5.2.4)
Remaining service gaps: Deeper exploitability confirmation (e.g., triggering vsFTPd backdoor) and host/config-level auditing beyond read-only recon/banner/service probes.

[Initial Probe - FTP Banner]: target=127.0.0.1 port=2121 state=open banner='220 (vsFTPd 2.3.4)'
[Initial Probe - Anon FTP]: target=127.0.0.1 port=2121 anonymous_login=ALLOWED welcome='220 (vsFTPd 2.3.4)' response='230 Login successful.' sample_entries=[]
[Initial Probe - HTTP]: url=http://127.0.0.1:8080/ status=200 server=Apache/2.2.8 (Ubuntu) DAV/2 x_powered_by=PHP/5.2.4-2ubuntu5.10 body_preview="<html><head><title>Metasploitable2 - Linux</title></head><body>\n<pre>\n\n                _                  _       _ _        _     _      ____  \n _ __ ___   ___| |_ __ _ ___ _ __ | | ___ (_) |_ __ _| |__ | | ___|___ \\ \n| '_ ` _ \\ / _ \\ 